In [61]:
import pandas as pd
import numpy as np
import tensorflow as tf


print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [62]:
train_file_path = "./train-data.tsv"
test_file_path = "./valid-data.tsv"

In [63]:
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'text'])

In [64]:
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'text'])

In [65]:
train_df.head()

,label,text
0,ham,ahhhh...just woken up!had a bad dream about u ...
1,ham,you can never do nothing
2,ham,"now u sound like manky scouse boy steve,like! ..."
3,ham,mum say we wan to go then go... then she can s...
4,ham,never y lei... i v lazy... got wat? dat day ü ...


$$
\text{Using HuggingFace sentence transformer}
$$
https://huggingface.co/models?pipeline_tag=sentence-similarity&sort=trending

In [66]:
print(train_df.columns)

Index(['label', 'text'], dtype='object')


In [67]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder

sentence_model = SentenceTransformer('paraphrase-MiniLM-L6-v2', device='cuda')
def preprocess(dataframe):

    texts = dataframe['text'].tolist()

    embeddings = sentence_model.encode(texts, batch_size=64, show_progress_bar=True)

    dataframe['encoded'] = list(embeddings)

    label_encoder = LabelEncoder()
    dataframe['label_encoded'] = label_encoder.fit_transform(dataframe['label'])

    dataframe = dataframe.drop(['label', 'text'], axis=1)
    dataframe = dataframe.rename(columns={'label_encoded': 'label', 'encoded': 'text'})
    dataframe.attrs['label_encoder'] = label_encoder

    return dataframe



In [68]:
train = preprocess(train_df)

Batches:   0%|          | 0/66 [00:00<?, ?it/s]

In [69]:
valid = preprocess(test_df)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [70]:
train.head()

,text,label
0,"[0.31281447, -0.3585767, 0.0356886, 0.02793775...",0
1,"[0.33434135, -0.088207304, -0.10494735, -0.132...",0
2,"[0.5315699, -0.20541458, 0.15435672, -0.358831...",0
3,"[0.68175876, 0.15692922, -0.017596634, 0.08803...",0
4,"[-0.33380458, 0.043861836, -0.17576396, -0.275...",0


In [71]:
X_train = np.stack(train['text'].values)
y_train = train['label'].values

X_valid = np.stack(valid['text'].values)
y_valid = valid['label'].values

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

model = Sequential([
    Input(shape=(384,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [53]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [54]:
model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=5, batch_size=16)

Epoch 1/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9015 - loss: 0.2393 - val_accuracy: 0.9878 - val_loss: 0.0438
Epoch 2/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9887 - loss: 0.0315 - val_accuracy: 0.9878 - val_loss: 0.0395
Epoch 3/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9962 - loss: 0.0158 - val_accuracy: 0.9878 - val_loss: 0.0419
Epoch 4/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9977 - loss: 0.0082 - val_accuracy: 0.9899 - val_loss: 0.0472
Epoch 5/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9996 - loss: 0.0033 - val_accuracy: 0.9892 - val_loss: 0.0483


In [55]:
def encode(text):
    if text:
        text_encoded = sentence_model.encode(text, convert_to_tensor=False)
        return text_encoded
    else:
        raise ValueError("Text input cannot be empty!")


In [56]:
def predict_message(pred_text):
    encoded_text = encode(pred_text)
    encoded_text = np.expand_dims(encoded_text, axis=0)

    pred = model.predict(encoded_text, verbose=0)
    pred_value = pred[0][0]

    label = "spam" if pred_value > 0.5 else "ham"

    return [pred_value, label]


In [57]:
test = predict_message("our new mobile video service is live. just install on your phone to start watching.")

In [59]:
print(test)

[np.float32(0.27368417), 'ham']


In [60]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
    test_messages = ["how are you doing today",
                       "sale today! to stop texts call 98912460324",
                       "i dont want to go. can we try it a different day? available sat",
                       "our new mobile video service is live. just install on your phone to start watching.",
                       "you have won £1000 cash! call to claim your prize.",
                       "i'll bring it tomorrow. don't forget the milk.",
                       "wow, is your arm alright. that happened to me one time too"
                      ]

    test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
    passed = True
    output = []

    for msg, ans in zip(test_messages, test_answers):
        prediction = predict_message(msg)
        output.append(prediction[1])

        if prediction[1] != ans:
            passed = False

    print(output)
    if passed:
        print("You passed the challenge. Great job!")
    else:
        print("You haven't passed yet. Keep trying.")

test_predictions()


KeyboardInterrupt: 